# SQL Avançado Gold Sicredi

Este notebook apresenta exemplos de SQL avançado usando Databricks SQL, com base nas tabelas Gold do processo Sicredi. Os scripts cobrem CTEs, ROW_NUMBER, LAG/LEAD, RANK/DENSE_RANK, consultas de crescimento de clientes e classificação por quartis (NTILE). Cada consulta utiliza as tabelas `workspace.gold.gold_transacoes_cartao_particionado` e `workspace.gold.gold_resumo_cliente_mes_particionado`.


In [0]:
%sql
-- Exemplo de CTE (Common Table Expression)
-- Utiliza CTE para organizar a análise de transações por cliente
WITH transacoes_cliente AS (
    SELECT id_cliente, ano_mes, valor
    FROM workspace.gold.gold_transacoes_cartao_particionado
    WHERE valor > 0
)
SELECT id_cliente, ano_mes, SUM(valor) AS valor_total
FROM transacoes_cliente
GROUP BY id_cliente, ano_mes
ORDER BY valor_total DESC;


In [0]:
%sql
-- ROW_NUMBER para deduplicação/versionamento
-- Seleciona, para cada cliente, a transação mais recente de cada mês
SELECT *,
       ROW_NUMBER() OVER (PARTITION BY id_cliente, ano_mes ORDER BY data_transacao DESC) AS rn
FROM workspace.gold.gold_transacoes_cartao_particionado
QUALIFY rn = 1;


In [0]:
%sql
-- LAG/LEAD para movimentação entre meses
-- Compara valor total movimentado por cliente entre meses consecutivos
SELECT 
    id_cliente,
    ano_mes,
    valor_total,
    LAG(valor_total) OVER (PARTITION BY id_cliente ORDER BY ano_mes) AS valor_mes_anterior,
    LEAD(valor_total) OVER (PARTITION BY id_cliente ORDER BY ano_mes) AS valor_proximo_mes,
    (valor_total - LAG(valor_total) OVER (PARTITION BY id_cliente ORDER BY ano_mes)) AS variacao
FROM workspace.gold.gold_resumo_cliente_mes_particionado;


In [0]:
%sql
-- RANK/DENSE_RANK para rankings
-- Ranking de clientes por valor movimentado no mês
SELECT 
    id_cliente, ano_mes, valor_total,
    RANK() OVER (ORDER BY valor_total DESC) AS ranking_valor,
    DENSE_RANK() OVER (ORDER BY valor_total DESC) AS ranking_valor_dense
FROM workspace.gold.gold_resumo_cliente_mes_particionado;


In [0]:
%sql
-- Consulta: clientes com crescimento >50% entre dois meses consecutivos
-- Corrige filtragem usando uma subquery para calcular crescimento_relativo
SELECT *
FROM (
  SELECT 
    id_cliente, ano_mes, valor_total,
    LAG(valor_total) OVER (PARTITION BY id_cliente ORDER BY ano_mes) AS valor_mes_anterior,
    CASE
      WHEN LAG(valor_total) OVER (PARTITION BY id_cliente ORDER BY ano_mes) > 0 THEN
           (valor_total - LAG(valor_total) OVER (PARTITION BY id_cliente ORDER BY ano_mes)) / LAG(valor_total) OVER (PARTITION BY id_cliente ORDER BY ano_mes)
      ELSE NULL END AS crescimento_relativo
  FROM workspace.gold.gold_resumo_cliente_mes_particionado
) t
WHERE crescimento_relativo > 0.5;


In [0]:
%sql
-- Classificação de clientes em quartis usando NTILE
-- Distribui clientes em quartis conforme valor movimentado no mês
SELECT 
    id_cliente, ano_mes, valor_total,
    NTILE(4) OVER (ORDER BY valor_total DESC) AS quartil_valor_movimentado
FROM workspace.gold.gold_resumo_cliente_mes_particionado;
